# Finding similar texts: MiniLM embeddings and HNSW

A sentence embedding model turns a text into a vector so that texts with similar meaning point in similar
directions. Searching texts by meaning then becomes a nearest neighbour search among vectors, which
[HNSW](hnsw.ipynb) answers approximately, without comparing the query with every text.

- [minilm.py](minilm.py) runs [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
  a small transformer trained on about a billion pairs of related sentences. It does the steps of the
  sentence-transformers library itself: tokenize, run the model's ONNX export with onnxruntime, average the
  token vectors and normalise. Every text becomes 384 numbers, a vector of length 1.
- [corpora.py](corpora.py) collects the texts. The first set is the summaries of the docstrings in Python's
  standard library, numpy and matplotlib, read from their source files. The second is this repository's
  notes and commit messages.
- [hnsw.py](hnsw.py) indexes the vectors with cosine distance, 1 − cos, and finds the nearest ones to a query's
  vector.

The model, about 90 MB, is downloaded from Hugging Face on first use and cached.

In [ ]:
import os
try:
    # In Google Colab: install the model runtime, and clone the repository for its scripts and notes
    import google.colab
    %pip install -q onnxruntime tokenizers
    if not os.path.exists('hnsw.py'):
        !git clone -q https://github.com/tinkerersnotes/tinkerersnotes.git
        %cd tinkerersnotes/hnsw
except ImportError:
    pass

In [ ]:
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter

from corpora import python_docstrings, repo_notes
from hnsw import HNSW, exact_knn, recall
from minilm import MiniLM

SEED = 1
K = 10

BLUE, INK, MUTED, EDGE, GRID, SURFACE = '#2a78d6', '#0b0b0b', '#898781', '#c3c2b7', '#e1e0d9', '#fcfcfb'
plt.rcParams.update({
    'axes.edgecolor': EDGE, 'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.spines.top': False,
    'axes.spines.right': False, 'legend.frameon': False, 'lines.linewidth': 2,
})

model = MiniLM()

## From text to vector

The tokenizer splits text into word pieces from a fixed vocabulary; rare words become several pieces. `[CLS]` and
`[SEP]` mark the start and the end.

In [ ]:
print(model.tokens('Tokenizing unbelievably rare words'))
vector = model.encode('A man is eating food.')[0]
print(f'{vector.shape[0]} numbers, length {np.linalg.norm(vector):.3f}, first few {np.round(vector[:6], 3)}')

For vectors of length 1, the dot product is the cosine of the angle between them: 1 for the same direction,
around 0 for unrelated texts. Sentences on the same subject score high even with few words in common: 0.44 for the two
about putting a list in order, 0.76 for the two meals. Shared words alone count for less: the chores to sort out
share "list" and "sort" with the first question, and score 0.20.

In [ ]:
sentences = [
    'How do I sort a list in Python?',
    'Order the items of a sequence',
    'A list of chores to sort out before the trip',
    'A man is eating food.',
    'A man is eating a piece of bread.',
    'A man is riding a horse.',
]
vectors = model.encode(sentences)
similarity = vectors @ vectors.T

fig, ax = plt.subplots(figsize=(8.5, 6))
ramp = LinearSegmentedColormap.from_list('blue', [SURFACE, '#9ec5f4', BLUE, '#104281'])
image = ax.imshow(similarity, cmap=ramp, vmin=0, vmax=1)
for (i, j), value in np.ndenumerate(similarity):
    ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=10,
            color='white' if value > 0.6 else INK)
ax.set_xticks(range(len(sentences)), [str(i + 1) for i in range(len(sentences))])
ax.set_yticks(range(len(sentences)), [f'{i + 1}. {s}' for i, s in enumerate(sentences)])
ax.grid(False)
ax.set_title('Cosine similarity of MiniLM embeddings', color=INK)
fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()

## Docstrings of Python's standard library, numpy and matplotlib

Every public module, class and function with a docstring contributes its qualified name and the docstring's
first paragraph. The source files are parsed, not imported. The standard library's datetime classes live in
`_pydatetime`, its pure Python version, so they appear under that name.

In [ ]:
docs = python_docstrings()
print(f'{len(docs)} texts:', ', '.join(f'{origin} {count}' for origin, count in Counter(
    'numpy' if name.startswith('numpy') else 'matplotlib' if name.startswith('matplotlib')
    else 'standard library' for name, _ in docs).most_common()))
for _, text in docs[::1500]:
    print(' ', text[:120])

start = time.perf_counter()
doc_vectors = model.encode([text for _, text in docs])
print(f'Embedded in {time.perf_counter() - start:.0f} s')

### The index, and how close to exact it gets

300 texts are held out as queries, and the rest are indexed. Recall@10 is measured against brute force for a
range of `ef`. Then the held-out texts are added too: an HNSW index takes insertions at any time.

In [ ]:
held_out = np.random.default_rng(SEED).choice(len(docs), 300, replace=False)
indexed = np.setdiff1d(np.arange(len(docs)), held_out)

index = HNSW(model.dim, metric='cosine', seed=SEED)
start = time.perf_counter()
index.add_items(doc_vectors[indexed])
print(f'Indexed {len(index)} vectors in {time.perf_counter() - start:.0f} s, {index.top + 1} layers')

truth, _ = exact_knn(doc_vectors[indexed], doc_vectors[held_out], K, metric='cosine')
rows = []
for ef in (10, 20, 40, 80, 160):
    index.distance_count = 0
    start = time.perf_counter()
    found, _ = index.search_many(doc_vectors[held_out], K, ef)
    rows.append((ef, recall(found, truth), index.distance_count / len(held_out)))
    print(f'ef {ef:4}: recall@{K} {rows[-1][1]:.3f}, {rows[-1][2]:5.0f} distances, '
          f'{1000 * (time.perf_counter() - start) / len(held_out):.2f} ms per query')
rows = np.array(rows)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(rows[:, 2], rows[:, 1], '-o', color=BLUE, markersize=6)
for ef, r, d in rows:
    ax.annotate(f'ef {ef:.0f}', (d, r), textcoords='offset points', xytext=(6, -12), color=MUTED, fontsize=9)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.xaxis.set_minor_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}' if str(int(x))[0] in '25' else ''))
ax.set_xlabel(f'Distances computed per query (brute force: {len(index):,})')
ax.set_ylabel(f'Recall@{K}')
ax.set_title(f'HNSW on {len(index):,} MiniLM embeddings, 384 dimensions', color=INK)
plt.show()

# Now index the held-out texts as well; ids follow the order of insertion
index.add_items(doc_vectors[held_out])
doc_order = np.concatenate([indexed, held_out])

### Asking in plain words

The query is embedded like any text, and the index returns the texts closest in meaning. The similarity shown is
the cosine, 1 minus the distance. Most answers are what a person would point to, though the query rarely shares
their words. The thousands separators query shows a limit of indexing only summaries: `locale.format_string` does
group thousands, but says so in its docstring's second paragraph, so its summary ranks only 17th.

In [ ]:
def ask(query, index, texts, order, k=4, ef=40, width=120):
    ids, distances = index.search(model.encode(query)[0], k, ef)
    print(query)
    for i, d in zip(ids, distances):
        print(f'  {1 - d:.2f}  {texts[order[i]][1][:width]}')
    print()

for query in ['parse a date from a string',
              'read a compressed zip archive',
              'make a random password',
              'wait until all threads are finished',
              'measure how long code takes to run',
              'find files whose names match a pattern',
              'solve a linear system of equations',
              'save a figure to an image file',
              'format a number with thousands separators',
              'turn a list of lists into one flat list']:
    ask(query, index, docs, doc_order)

## This repository's notes

The same search over this repository's Markdown files, the Markdown cells of its notebooks, and its commit
messages. A long paragraph is cut at 256 tokens, and the model was trained on texts of up to 128, so it gets
the gist of a paragraph from its start. With a few hundred texts brute force would do; the index is the same
code either way.

Short paragraphs match on their few words. A one-line lead-in to a list, "Divide and conquer, already tested, was
the reference:", ranks above the paragraph that introduces the search for prior work.

In [ ]:
notes = repo_notes('..')
print(f'{len(notes)} texts:', ', '.join(f'{source} {count}' for source, count in Counter(
    'commits' if source.startswith('commit') else source for source, _ in notes).most_common()))

note_vectors = model.encode([text for _, text in notes])
note_index = HNSW(model.dim, metric='cosine', seed=SEED)
note_index.add_items(note_vectors)
note_order = np.arange(len(notes))
labelled = [(source, f'[{source}] {text}') for source, text in notes]

for query in ['why did the zip get stuck in the merge',
              'run machine code from a memory mapped file',
              'seam carving in the browser with threads',
              'how to open the notebooks in Google Colab',
              'has anyone published weighted divide and conquer before']:
    ask(query, note_index, labelled, note_order, k=3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tinkerersnotes/tinkerersnotes/blob/main/hnsw/minilm_search.ipynb)